[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/A3_Recv/blob/main/main_notebook.ipynb)

# DINOv3 ConvNeXt Base — clasificador lineal y ajuste parcial

Modelo: **`facebook/dinov3-convnext-base-pretrain-lvd1689m`**. Arquitectura visual CNN; pesos destilados previamente de un profesor ViT-7B. Aquí no se carga ni se ejecuta ese profesor, ni se repite el preentrenamiento DINO. Entrenamos con las etiquetas de `train.npz` y seleccionamos checkpoints solo con `validation.npz`.

| Fase | Parámetros que entrenamos | CSV |
|---|---|---|
| Linear probe | Nueva cabeza lineal de 10 clases; CNN congelada | `submission_dinov3_linear_256.csv` |
| Ajuste parcial | Cabeza ya entrenada + última etapa convolucional + normalización final | `submission_dinov3_partial_ft_256.csv` |

Se conserva el experimento anterior en [resnet18_augmix.ipynb](https://github.com/rivas-vega/A3_Recv/blob/main/resnet18_augmix.ipynb). Referencias comunicadas por Pedro: **61,093%** para ResNet baseline y **75,796%** para ResNet + AugMix en Kaggle. No son resultados de DINOv3.

Selecciona una GPU A100 y ejecuta las celdas en orden. Este notebook es autocontenido. Guarda cachés, checkpoints y CSV en una carpeta nueva de Drive, independiente de los resultados ResNet. No envía submissions automáticamente.

## Por qué la entrada es 256 × 256

No heredamos los 224 de ResNet ni los 128 del experimento anterior. El [README oficial de DINOv3](https://github.com/facebookresearch/dinov3/blob/6876159a11b4df116f30f667f8c9888617df0751/README.md#image-transform) recomienda, para los pesos web LVD-1689M, **redimensionado cuadrado a 256, bilinear con antialias, conversión a float [0,1] y normalización ImageNet**. Se aplica idénticamente al extraer características, ajustar el modelo y predecir.

El preentrenamiento no tiene una única resolución universal para todos sus modelos y fases. El paper evalúa específicamente los ConvNeXt a 256 y 512 (sección 7.2). La [receta pública de ejemplo de destilación ConvNeXt](https://github.com/facebookresearch/dinov3/blob/6876159a11b4df116f30f667f8c9888617df0751/dinov3/configs/train/distillation_convnext/multi_distillation_convnext_test.yaml) usa crops globales de 512 para el profesor y factor profesor/alumno 2; el código reduce esos crops a 256 para el alumno. Es una receta de ejemplo, no una prueba de toda la historia de entrenamiento de los pesos publicados.

Usamos **32 → 256 (×8)** sin recorte central adicional de evaluación. La ampliación no reconstruye detalle. No aplicamos un segundo procesador de Hugging Face que pueda redimensionar otra vez a 224. La configuración declarada por el procesador se registra para poder contrastarla con nuestra elección explícita basada en el README.

Fuentes: [DINOv3, sección 7.2](https://arxiv.org/html/2508.10104v1#S7.SS2), [ficha del checkpoint](https://huggingface.co/facebook/dinov3-convnext-base-pretrain-lvd1689m).

## 1. Acceso a los pesos y configuración

**Antes de ejecutar:** abre [el checkpoint en Hugging Face](https://huggingface.co/facebook/dinov3-convnext-base-pretrain-lvd1689m) con tu cuenta y completa la solicitud/aceptación de acceso. Espera a que el acceso esté concedido. En [Hugging Face → Access Tokens](https://huggingface.co/settings/tokens), crea un token de tipo **Read** (lectura), o uno fine-grained con permiso de lectura para ese modelo gated y pégalo entre las comillas de **`HF_TOKEN = ""`** en la celda dedicada, justo antes de cargar el checkpoint. Ejecuta esa celda; no aparecerá ninguna petición de entrada en la salida.

El token se escribe únicamente en tu copia privada de Colab. Bórralo antes de compartirla o guardarla en GitHub; el repositorio contiene la casilla vacía. Un **403 / «not in the authorized list»** requiere obtener acceso al modelo con la misma cuenta que creó el token: cambiar el modo de introducirlo no concede ese permiso.

**Comprobación automática:** al cargar el checkpoint, primero se verifica el token con `whoami()` y se muestra solo el nombre de la cuenta. Después `auth_check()` comprueba el acceso efectivo al modelo. Si aparece **«Acceso de lectura a DINOv3: OK»**, ambas comprobaciones han pasado. Un 401 durante la descarga, sin estas comprobaciones, no basta para distinguir un token rechazado de un permiso de modelo ausente. Si cambias el token, vuelve a ejecutar su celda y la carga del checkpoint.

Para Kaggle reutiliza `KAGGLE_API_TOKEN` o el login interactivo. El notebook falla con una explicación si los pesos no son accesibles: nunca los sustituye por una red aleatoria.

Configuración propuesta (hiperparámetros iniciales, no óptimos demostrados):

- **Linear probe:** hasta 100 épocas de una cabeza pequeña sobre características cacheadas; LR 1e-3, paciencia 10 desde época 10. La CNN solo se ejecuta para extraer características. Train incluye imagen completa y su volteo horizontal; val y test, solo imagen completa.
- **Ajuste parcial:** hasta 10 épocas, LR 1e-5 en última etapa/normalización y 1e-4 en cabeza; calentamiento de 1 época, descenso coseno, paciencia 3 desde época 4. Primeras tres etapas congeladas. Crop/flip suaves a 32px + AugMix severidad 2 + JSD con peso 6.
- **Presupuesto:** hasta aproximadamente 3 h de ajuste parcial por ejecución, comprobado entre épocas. La preparación, extracción de características, cabeza lineal e inferencia son adicionales. El tiempo real se mide; no se promete una duración fija.

Los dos CSV comparan protocolos de adaptación; el segundo introduce también AugMix, así que no aísla únicamente el efecto de descongelar capas. La selección por validación CIFAR-10 no garantiza la mejor generalización al test sintético.

In [ ]:
%pip -q install "transformers==4.57.3" "huggingface_hub>=0.34,<1.0" kagglehub safetensors

In [ ]:
from pathlib import Path
import os
import json
import gc
import copy
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision import transforms as T
from torchvision.transforms import v2, InterpolationMode
import transformers
from transformers import AutoModel, AutoImageProcessor
from huggingface_hub import HfApi
from IPython.display import display

if transformers.__version__ != '4.57.3':
    raise RuntimeError('Reinicia la sesión de Colab tras instalar transformers y vuelve a ejecutar las celdas.')
if not torch.cuda.is_available():
    raise RuntimeError('Selecciona una GPU en Entorno de ejecución → Cambiar tipo de entorno.')
DEVICE = torch.device('cuda')
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

USE_DRIVE = True
RUN_NAME = 'dinov3_convnext_base_256_v1'
NUM_WORKERS = min(4, os.cpu_count() or 1)
EVAL_BATCH_SIZE = 128
FT_HOURS = 3.0
MODEL_ID = 'facebook/dinov3-convnext-base-pretrain-lvd1689m'
assert FT_HOURS > 0
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = Path('/content/drive/MyDrive/A3_Recv/runs') / RUN_NAME
else:
    RUN_ROOT = Path('/content/recvis_runs') / RUN_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)

CFG = dict(
    code_version='dinov3_convnext_partial_v1', seed=42, model_id=MODEL_ID,
    feature_dim=1024, image_size=256,
    preprocessing=dict(size=[256, 256], interpolation='bilinear', antialias=True,
                       input_dtype='uint8', scale=1/255,
                       mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225],
                       implementation='torchvision_v2_ToImage_Resize_ToDtype_Normalize', center_crop=False),
    lp_epochs=100, lp_min_epochs=10, lp_patience=10, lp_batch_size=2048,
    lp_lr=1e-3, lp_weight_decay=1e-4, lp_train_views=['original', 'horizontal_flip'],
    ft_epochs=10, ft_min_epochs=4, ft_patience=3, ft_warmup_epochs=1,
    ft_trainable_stages=1, ft_batch_size=64, ft_backbone_lr=1e-5, ft_head_lr=1e-4,
    ft_weight_decay=1e-4, crop_padding=2, augmix_severity=2, augmix_width=3,
    jsd_weight=6.0, torch_version=str(torch.__version__),
    torchvision_version=str(torchvision.__version__), transformers_version=transformers.__version__,
    amp_dtype=str(AMP_DTYPE),
)
assert 1 <= CFG['ft_trainable_stages'] <= 4
print('GPU:', torch.cuda.get_device_name(0), '| memoria GB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print('Resultados:', RUN_ROOT)

## 2. Funciones del experimento

Estas celdas definen las funciones; todavía no entrenan. Los checkpoints se guardan al completar cada época. Si Colab se corta a mitad, esa época se repite. La caché se guarda por conjunto/vista: se reutilizan las vistas terminadas y se repite una extracción interrumpida.

El checkpoint del ajuste parcial guarda solo los parámetros modificables. Para reconstruirlo se cargan primero los pesos originales desde la **misma revisión inmutable de Hugging Face**, registrada en `config.json`. Se comprueba también la identidad de la cabeza lineal de partida. No se mezclan pesos nuevos con cachés antiguas.

In [ ]:
"""Functions embedded in the Colab notebook; no training is started on import."""
import csv
import gc
import hashlib
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def atomic_save(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    torch.save(value, temporary)
    os.replace(temporary, path)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def write_history(history, path):
    if history:
        with open(path, 'w', newline='') as handle:
            writer = csv.DictWriter(handle, fieldnames=list(history[0]))
            writer.writeheader()
            writer.writerows(history)


def cpu_state(module):
    return {key: value.detach().cpu().clone() for key, value in module.state_dict().items()}


def state_fingerprint(state):
    digest = hashlib.sha256()
    for name in sorted(state):
        value = state[name].detach().cpu().contiguous()
        digest.update(name.encode())
        digest.update(str(value.dtype).encode())
        digest.update(str(tuple(value.shape)).encode())
        digest.update(value.numpy().tobytes())
    return digest.hexdigest()


def is_better(accuracy, loss, best_accuracy, best_loss):
    return accuracy > best_accuracy or (accuracy == best_accuracy and loss < best_loss)


def lr_factor(epoch, maximum, warmup=1):
    if epoch <= warmup:
        return epoch / max(1, warmup)
    progress = (epoch - warmup) / max(1, maximum - warmup)
    return 0.1 + 0.9 * (1 + math.cos(math.pi * progress)) / 2

In [ ]:
class ImagesDataset(Dataset):
    def __init__(self, images, targets, transform, flip=False):
        assert len(images) == len(targets)
        self.images, self.targets, self.transform, self.flip = images, targets, transform, flip

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = Image.fromarray(self.images[index])
        if self.flip:
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        return self.transform(image), int(self.targets[index])


class AugMixDataset(Dataset):
    def __init__(self, images, labels, transform, basic_transform, augmix_transform, seed, epoch):
        self.images, self.labels, self.transform = images, labels, transform
        self.basic_transform, self.augmix_transform = basic_transform, augmix_transform
        self.seed, self.epoch = seed, epoch

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        # Isolate CPU RNG per image so an interrupted epoch can be repeated.
        with torch.random.fork_rng(devices=[]):
            torch.random.default_generator.manual_seed(self.seed + self.epoch * len(self.images) + index)
            image = self.basic_transform(Image.fromarray(self.images[index]))
            views = (self.transform(image), self.transform(self.augmix_transform(image)),
                     self.transform(self.augmix_transform(image)))
        return views, int(self.labels[index])


def make_loader(dataset, training=False, epoch=0):
    return DataLoader(
        dataset, batch_size=CFG['ft_batch_size'] if training else EVAL_BATCH_SIZE,
        shuffle=training, drop_last=False, num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == 'cuda', persistent_workers=False,
        generator=torch.Generator().manual_seed(CFG['seed'] + epoch),
    )


def embedding(backbone, images):
    # The ConvNeXt global descriptor is normalized global-average pooling, not a ViT CLS token.
    outputs = backbone(pixel_values=images)
    features = outputs.pooler_output
    if features.ndim != 2 or features.shape[1] != CFG['feature_dim']:
        raise ValueError(f'Unexpected DINOv3 feature shape: {features.shape}')
    return features


def cache_metadata(key, length):
    return dict(key=key, rows=length, model_id=CFG['model_id'], revision=CFG['model_revision'],
                preprocessing=CFG['preprocessing'], data_sha256=CFG['data_sha256'],
                feature_dim=CFG['feature_dim'], torch_version=CFG['torch_version'],
                torchvision_version=CFG['torchvision_version'], transformers_version=CFG['transformers_version'],
                amp_dtype=CFG['amp_dtype'])


@torch.inference_mode()
def extract_features(backbone, images, key, transform, flip=False):
    path = RUN_ROOT / 'features' / f'{key}.pt'
    path.parent.mkdir(parents=True, exist_ok=True)
    metadata = cache_metadata(key, len(images))
    metadata['flip'] = flip
    if path.exists():
        cached = torch.load(path, map_location='cpu', weights_only=True)
        if cached['metadata'] != metadata:
            raise ValueError('Feature cache incompatible; use a new RUN_NAME.')
        features = cached['features']
        if features.shape != (len(images), CFG['feature_dim']) or not torch.isfinite(features).all():
            raise ValueError('Feature cache is incomplete or contains non-finite values.')
        print(f'{key}: cache recuperada ({len(features):,} imágenes).')
        return features
    if any(parameter.requires_grad for parameter in backbone.parameters()):
        raise ValueError('Feature extraction must use the original frozen backbone.')
    backbone.eval()
    loader = make_loader(ImagesDataset(images, np.arange(len(images)), transform, flip=flip))
    features = torch.empty(len(images), CFG['feature_dim'], dtype=torch.float32)
    cursor, started = 0, time.monotonic()
    for step, (batch, indices) in enumerate(loader, 1):
        expected = torch.arange(cursor, cursor + len(indices))
        if not torch.equal(indices, expected):
            raise ValueError('The feature cache must preserve dataset order.')
        batch = batch.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == 'cuda'):
            values = embedding(backbone, batch)
        features[cursor:cursor + len(indices)] = values.cpu().float()
        cursor += len(indices)
        if step == 1 or step % 100 == 0 or cursor == len(images):
            print(f'{key}: {cursor:,}/{len(images):,} | {(time.monotonic() - started)/60:.1f} min', flush=True)
    if cursor != len(images) or not torch.isfinite(features).all():
        raise ValueError('Invalid feature extraction; cache was not saved.')
    atomic_save({'metadata': metadata, 'features': features}, path)
    return features

In [ ]:
@torch.inference_mode()
def evaluate_head(head, features, labels):
    head.eval()
    loss_sum = correct = 0
    for start in range(0, len(labels), CFG['lp_batch_size']):
        x = features[start:start + CFG['lp_batch_size']].to(DEVICE, dtype=torch.float32)
        y = labels[start:start + CFG['lp_batch_size']].to(DEVICE)
        logits = head(x)
        loss_sum += F.cross_entropy(logits, y, reduction='sum').item()
        correct += (logits.argmax(1) == y).sum().item()
    return loss_sum / len(labels), correct / len(labels)


def train_linear(train_features, train_labels, val_features, val_labels):
    seed_everything(CFG['seed'])
    folder = RUN_ROOT / 'linear'
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / 'last.pt'
    head = nn.Linear(CFG['feature_dim'], 10).to(DEVICE)
    nn.init.normal_(head.weight, std=0.01)
    nn.init.zeros_(head.bias)
    optimizer = torch.optim.AdamW(head.parameters(), lr=CFG['lp_lr'], weight_decay=CFG['lp_weight_decay'])
    history, best_state = [], None
    best_accuracy, best_loss, best_epoch, bad_epochs = -1.0, float('inf'), 0, 0
    start_epoch, finished, stop_reason = 1, False, 'running'
    if path.exists():
        checkpoint = torch.load(path, map_location='cpu', weights_only=True)
        if checkpoint['config'] != CFG:
            raise ValueError('Linear checkpoint incompatible; use a new RUN_NAME.')
        head.load_state_dict(checkpoint['head'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        history, best_state = checkpoint['history'], checkpoint['best_head']
        best_accuracy, best_loss, best_epoch = checkpoint['best_accuracy'], checkpoint['best_loss'], checkpoint['best_epoch']
        bad_epochs, finished = checkpoint['bad_epochs'], checkpoint['finished']
        start_epoch, stop_reason = checkpoint['epoch'] + 1, checkpoint['stop_reason']
        del checkpoint
        print(f'Linear: recuperada época {start_epoch - 1}.')
    if not finished:
        # Copies leave inference_mode tensors behind and allow head backpropagation.
        x_train = train_features.to(device=DEVICE, dtype=torch.float32, copy=True)
        y_train = train_labels.to(device=DEVICE, dtype=torch.long)
        x_val = val_features.to(device=DEVICE, dtype=torch.float32, copy=True)
        y_val = val_labels.to(device=DEVICE, dtype=torch.long)
        for epoch in range(start_epoch, CFG['lp_epochs'] + 1):
            started = time.monotonic()
            head.train()
            optimizer.param_groups[0]['lr'] = CFG['lp_lr'] * lr_factor(epoch, CFG['lp_epochs'])
            generator = torch.Generator(device=DEVICE).manual_seed(CFG['seed'] + epoch)
            order = torch.randperm(len(y_train), device=DEVICE, generator=generator)
            loss_sum = correct = 0
            for indices in order.split(CFG['lp_batch_size']):
                x, y = x_train[indices], y_train[indices]
                optimizer.zero_grad(set_to_none=True)
                logits = head(x)
                loss = F.cross_entropy(logits, y)
                if not torch.isfinite(loss):
                    raise FloatingPointError('Non-finite linear loss.')
                loss.backward()
                optimizer.step()
                loss_sum += loss.item() * len(y)
                correct += (logits.argmax(1) == y).sum().item()
            val_loss, val_accuracy = evaluate_head(head, x_val, y_val)
            if is_better(val_accuracy, val_loss, best_accuracy, best_loss):
                best_state, best_accuracy, best_loss, best_epoch = cpu_state(head), val_accuracy, val_loss, epoch
                bad_epochs = 0
            else:
                bad_epochs += 1
            history.append(dict(epoch=epoch, train_loss=loss_sum / len(y_train),
                                train_accuracy=correct / len(y_train), val_loss=val_loss,
                                val_accuracy=val_accuracy, epoch_seconds=time.monotonic() - started))
            finished = epoch == CFG['lp_epochs'] or (epoch >= CFG['lp_min_epochs'] and bad_epochs >= CFG['lp_patience'])
            stop_reason = 'early_stopping' if finished and epoch < CFG['lp_epochs'] else ('max_epochs' if finished else 'running')
            atomic_save(dict(config=CFG, head=cpu_state(head), optimizer=optimizer.state_dict(),
                             epoch=epoch, history=history, best_head=best_state, best_epoch=best_epoch,
                             best_accuracy=best_accuracy, best_loss=best_loss, bad_epochs=bad_epochs,
                             finished=finished, stop_reason=stop_reason), path)
            write_history(history, folder / 'history.csv')
            print(f'Linear {epoch:03d} | val {val_accuracy:.2%} | loss {val_loss:.4f}', flush=True)
            if finished:
                break
        del x_train, y_train, x_val, y_val
    if best_state is None:
        raise RuntimeError('No valid linear checkpoint.')
    head.load_state_dict(best_state)
    head.eval()
    atomic_save(dict(config=CFG, head=best_state, epoch=best_epoch,
                     val_accuracy=best_accuracy, val_loss=best_loss), folder / 'best.pt')
    write_history(history, folder / 'history.csv')
    summary = dict(method='dinov3_linear', epochs_trained=len(history), best_epoch=best_epoch,
                   val_accuracy=best_accuracy, val_loss=best_loss,
                   training_hours=sum(row['epoch_seconds'] for row in history) / 3600,
                   stop_reason=stop_reason, completed=finished)
    (folder / 'summary.json').write_text(json.dumps(summary, indent=2))
    del optimizer
    return head, summary

In [ ]:
class DinoClassifier(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone, self.head = backbone, head

    def forward(self, images):
        features = embedding(self.backbone, images).float()
        # Same head precision as the cached linear probe, including at epoch 0.
        with torch.autocast(features.device.type, enabled=False):
            return self.head(features)


def configure_partial_finetuning(model):
    # Hugging Face DINOv3 ConvNeXt v4.57.3: four stages + final layer_norm.
    if len(model.backbone.stages) != 4:
        raise ValueError('Expected four ConvNeXt stages.')
    if not 1 <= CFG['ft_trainable_stages'] <= 4:
        raise ValueError('ft_trainable_stages must be between 1 and 4.')
    model.backbone.requires_grad_(False)
    for stage in model.backbone.stages[-CFG['ft_trainable_stages']:]:
        stage.requires_grad_(True)
    model.backbone.layer_norm.requires_grad_(True)
    model.head.requires_grad_(True)
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def trainable_state(model):
    # Frozen weights are recovered from the immutable original Hugging Face revision.
    return {name: parameter.detach().cpu().clone()
            for name, parameter in model.named_parameters() if parameter.requires_grad}


def load_trainable_state(model, state):
    parameters = dict(model.named_parameters())
    expected = {name for name, parameter in parameters.items() if parameter.requires_grad}
    if set(state) != expected:
        raise ValueError('Checkpoint trainable parameters do not match the current configuration.')
    with torch.no_grad():
        for name, value in state.items():
            if value.shape != parameters[name].shape:
                raise ValueError(f'Checkpoint shape mismatch: {name}')
            parameters[name].copy_(value)


def js_divergence(logits):
    log_prob = torch.stack([F.log_softmax(value.float(), dim=1) for value in logits])
    log_mean = torch.logsumexp(log_prob, dim=0) - math.log(len(logits))
    return (log_prob.exp() * (log_prob - log_mean.unsqueeze(0))).sum(-1).mean()


@torch.inference_mode()
def evaluate_images(model, loader):
    model.eval()
    total_loss = correct = count = 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == 'cuda'):
            logits = model(images)
        total_loss += F.cross_entropy(logits.float(), labels, reduction='sum').item()
        correct += (logits.argmax(1) == labels).sum().item()
        count += len(labels)
    return total_loss / count, correct / count


def finetune_epoch(model, loader, optimizer, scaler):
    model.train()
    # Keep frozen stages in eval mode; ConvNeXt uses LayerNorm, not BatchNorm.
    for stage in model.backbone.stages[:-CFG['ft_trainable_stages']]:
        stage.eval()
    sums = np.zeros(3, dtype=np.float64)
    correct = count = 0
    for views, labels in loader:
        images = torch.cat(views, dim=0).to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == 'cuda'):
            logits = model(images)
        clean, first, second = logits.split(len(labels))
        ce = F.cross_entropy(clean.float(), labels)
        jsd = js_divergence((clean, first, second))
        loss = ce + CFG['jsd_weight'] * jsd
        if not torch.isfinite(loss):
            raise FloatingPointError('Non-finite fine-tuning loss; previous complete checkpoint retained.')
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()
        sums += np.asarray([loss.item(), ce.item(), jsd.item()]) * len(labels)
        correct += (clean.argmax(1) == labels).sum().item()
        count += len(labels)
    return dict(train_loss=float(sums[0] / count), train_ce=float(sums[1] / count),
                train_jsd=float(sums[2] / count), train_accuracy=correct / count)


def finetune(model, train_images, train_labels, val_loader, transform, basic_transform, augmix_transform):
    seed_everything(CFG['seed'])
    count = configure_partial_finetuning(model)
    print(f'Fine-tuning: {count:,} parámetros entrenables; primeras etapas congeladas.')
    folder = RUN_ROOT / 'partial_ft'
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / 'last.pt'
    backbone_parameters = [(name, p) for name, p in model.backbone.named_parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW([
        {'params': [p for name, p in backbone_parameters if p.ndim > 1], 'lr': CFG['ft_backbone_lr'], 'weight_decay': CFG['ft_weight_decay']},
        {'params': [p for name, p in backbone_parameters if p.ndim <= 1], 'lr': CFG['ft_backbone_lr'], 'weight_decay': 0.0},
        {'params': model.head.parameters(), 'lr': CFG['ft_head_lr'], 'weight_decay': CFG['ft_weight_decay']},
    ])
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda' and AMP_DTYPE == torch.float16)
    history, best_state = [], None
    start_epoch, best_epoch, bad_epochs, finished, stop_reason = 1, 0, 0, False, 'running'
    initial_loss = initial_accuracy = None
    lp_checkpoint = torch.load(RUN_ROOT / 'linear' / 'best.pt', map_location='cpu', weights_only=True)
    if lp_checkpoint['config'] != CFG:
        raise ValueError('Linear probe configuration mismatch.')
    lp_digest = state_fingerprint(lp_checkpoint['head'])
    del lp_checkpoint
    if path.exists():
        checkpoint = torch.load(path, map_location='cpu', weights_only=True)
        if checkpoint['config'] != CFG or checkpoint['linear_sha256'] != lp_digest:
            raise ValueError('Fine-tuning checkpoint incompatible with this linear probe or configuration.')
        load_trainable_state(model, checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        scaler.load_state_dict(checkpoint['scaler'])
        history, best_state = checkpoint['history'], checkpoint['best_model']
        best_accuracy, best_loss, best_epoch = checkpoint['best_accuracy'], checkpoint['best_loss'], checkpoint['best_epoch']
        initial_accuracy, initial_loss = checkpoint['initial_accuracy'], checkpoint['initial_loss']
        bad_epochs, finished = checkpoint['bad_epochs'], checkpoint['finished']
        start_epoch, stop_reason = checkpoint['epoch'] + 1, checkpoint['stop_reason']
        torch.set_rng_state(checkpoint['torch_rng'])
        if DEVICE.type == 'cuda':
            torch.cuda.set_rng_state_all(checkpoint['cuda_rng'])
        del checkpoint
        print(f'Fine-tuning: recuperada época {start_epoch - 1}.')
    else:
        # Epoch 0 is an eligible checkpoint: never discard the unmodified linear model.
        initial_loss, initial_accuracy = evaluate_images(model, val_loader)
        best_accuracy, best_loss, best_state = initial_accuracy, initial_loss, trainable_state(model)
        print(f'Antes del fine-tuning: val {initial_accuracy:.2%}. Se conserva como época 0.')
    started, previous_seconds = time.monotonic(), 0.0
    if not finished:
        for epoch in range(start_epoch, CFG['ft_epochs'] + 1):
            if epoch > start_epoch and time.monotonic() - started + 1.15 * previous_seconds > FT_HOURS * 3600:
                stop_reason = 'time_budget'
                print('Pausa por presupuesto de tiempo. Reejecuta esta celda para continuar.')
                break
            epoch_started = time.monotonic()
            for group, base_lr in zip(optimizer.param_groups, (CFG['ft_backbone_lr'], CFG['ft_backbone_lr'], CFG['ft_head_lr'])):
                group['lr'] = base_lr * lr_factor(epoch, CFG['ft_epochs'], CFG['ft_warmup_epochs'])
            dataset = AugMixDataset(train_images, train_labels, transform, basic_transform, augmix_transform, CFG['seed'], epoch)
            loader = make_loader(dataset, training=True, epoch=epoch)
            metrics = finetune_epoch(model, loader, optimizer, scaler)
            val_loss, val_accuracy = evaluate_images(model, val_loader)
            if is_better(val_accuracy, val_loss, best_accuracy, best_loss):
                best_state, best_accuracy, best_loss, best_epoch = trainable_state(model), val_accuracy, val_loss, epoch
                bad_epochs = 0
            else:
                bad_epochs += 1
            history.append(dict(epoch=epoch, **metrics, val_loss=val_loss, val_accuracy=val_accuracy,
                                epoch_seconds=time.monotonic() - epoch_started))
            finished = epoch == CFG['ft_epochs'] or (epoch >= CFG['ft_min_epochs'] and bad_epochs >= CFG['ft_patience'])
            stop_reason = 'early_stopping' if finished and epoch < CFG['ft_epochs'] else ('max_epochs' if finished else 'running')
            atomic_save(dict(config=CFG, linear_sha256=lp_digest, model=trainable_state(model),
                             optimizer=optimizer.state_dict(), scaler=scaler.state_dict(),
                             epoch=epoch, history=history, best_model=best_state,
                             best_accuracy=best_accuracy, best_loss=best_loss, best_epoch=best_epoch,
                             initial_accuracy=initial_accuracy, initial_loss=initial_loss,
                             bad_epochs=bad_epochs, finished=finished, stop_reason=stop_reason,
                             torch_rng=torch.get_rng_state(),
                             cuda_rng=torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else None), path)
            previous_seconds = time.monotonic() - epoch_started
            history[-1]['epoch_seconds'] = previous_seconds
            write_history(history, folder / 'history.csv')
            print(f'FT {epoch:02d} | val {val_accuracy:.2%} | mejor época {best_epoch}'
                  f' | {previous_seconds/60:.1f} min', flush=True)
            if epoch == start_epoch:
                print(f"Tiempo restante estimado: {(CFG['ft_epochs'] - epoch)*previous_seconds/3600:.2f} h (aproximado).")
            del loader
            if finished:
                break
    load_trainable_state(model, best_state)
    model.eval()
    write_history(history, folder / 'history.csv')
    atomic_save(dict(config=CFG, model=best_state, linear_sha256=lp_digest,
                     epoch=best_epoch, val_accuracy=best_accuracy, val_loss=best_loss), folder / 'best.pt')
    summary = dict(method='dinov3_partial_ft', epochs_trained=len(history), best_epoch=best_epoch,
                   val_accuracy=best_accuracy, val_loss=best_loss, initial_val_accuracy=initial_accuracy,
                   training_hours=sum(row['epoch_seconds'] for row in history)/3600,
                   stop_reason=stop_reason, completed=finished)
    (folder / 'summary.json').write_text(json.dumps(summary, indent=2))
    del optimizer, scaler, best_state
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return summary

In [ ]:
def write_submission(ids, predictions, template_path, output_path):
    ids_array, predictions_array = np.asarray(ids), np.asarray(predictions)
    if ids_array.ndim != 1 or predictions_array.ndim != 1:
        raise ValueError('Submission IDs and labels must be vectors.')
    if not np.issubdtype(ids_array.dtype, np.integer) or not np.issubdtype(predictions_array.dtype, np.integer):
        raise ValueError('Submission IDs and labels must be integers.')
    ids, predictions = ids_array.tolist(), predictions_array.tolist()
    if len(ids) != len(predictions) or len(set(ids)) != len(ids):
        raise ValueError('Submission ID duplication or length mismatch.')
    if not predictions or not all(0 <= value <= 9 for value in predictions):
        raise ValueError('Predictions must be in [0,9].')
    with open(template_path, newline='') as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != ['id', 'label']:
            raise ValueError('The sample submission must have columns id,label.')
        ordered_ids = [int(row['id']) for row in reader]
    if len(ordered_ids) != len(ids) or len(set(ordered_ids)) != len(ordered_ids) or set(ordered_ids) != set(ids):
        raise ValueError('Test IDs and sample submission IDs differ.')
    lookup = dict(zip(ids, predictions))
    temporary = Path(str(output_path) + '.tmp')
    with open(temporary, 'w', newline='') as handle:
        writer = csv.writer(handle)
        writer.writerow(['id', 'label'])
        writer.writerows((identifier, lookup[identifier]) for identifier in ordered_ids)
    os.replace(temporary, output_path)
    print(f'{Path(output_path).name}: {len(ids):,} predicciones verificadas.')


@torch.inference_mode()
def predict_features(head, features):
    head.eval()
    predictions = []
    for start in range(0, len(features), CFG['lp_batch_size']):
        x = features[start:start + CFG['lp_batch_size']].to(DEVICE, dtype=torch.float32)
        predictions.extend(head(x).argmax(1).cpu().tolist())
    return np.asarray(predictions, dtype=np.int64)


@torch.inference_mode()
def predict_images(model, loader):
    model.eval()
    ids, predictions = [], []
    for images, batch_ids in loader:
        images = images.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == 'cuda'):
            logits = model(images)
        predictions.extend(logits.argmax(1).cpu().tolist())
        ids.extend(batch_ids.tolist())
    return np.asarray(ids, dtype=np.int64), np.asarray(predictions, dtype=np.int64)

In [ ]:
# Pega tu token entre las comillas y ejecuta esta celda. No genera ninguna salida.
# Esta celda debe ejecutarse también si cambias de token o reinicias la sesión.
# Borra el token antes de compartir esta copia o guardarla en GitHub.
HF_TOKEN = ""


In [ ]:
def colab_secret(name):
    from google.colab import userdata
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        return None

from huggingface_hub.utils import HfHubHTTPError
from requests.exceptions import RequestException

def verify_hf_access(token, model_id):
    # No imprime, guarda ni sustituye el token por credenciales de otra cuenta.
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError('Pega tu token en HF_TOKEN y ejecuta esa celda antes de continuar.')
    token = token.strip()
    if not token.startswith('hf_') or any(char.isspace() for char in token):
        raise RuntimeError('Pega el token completo que empieza por hf_, sin espacios dentro. No uses el nombre del token.')
    api = HfApi(token=token)
    try:
        identity = api.whoami()
    except HfHubHTTPError as exc:
        status = getattr(exc.response, 'status_code', None)
        if status in (401, 403):
            raise RuntimeError(
                'Hugging Face no acepta este token. Puede estar mal copiado, revocado o no ser válido. '
                'Crea uno en https://huggingface.co/settings/tokens, pégalo completo en HF_TOKEN '
                'y vuelve a ejecutar esa celda. Aún no se ha comprobado el acceso a DINOv3.'
            ) from None
        raise RuntimeError(f'No se pudo comprobar la identidad HF (HTTP {status}); reintenta más tarde.') from None
    except RequestException:
        raise RuntimeError('No se pudo conectar a Hugging Face. Reintenta la comprobación.') from None
    username = identity.get('name', '(sin nombre)')
    print('Token reconocido. Cuenta HF:', username)
    try:
        api.auth_check(repo_id=model_id, repo_type='model')
    except HfHubHTTPError as exc:
        status = getattr(exc.response, 'status_code', None)
        if status in (401, 403, 404):
            raise RuntimeError(
                f'El token identifica a {username}, pero no permite acceder a {model_id}. '
                f'Abre https://huggingface.co/{model_id} con ESA cuenta y comprueba que el acceso está aprobado. '
                'Revisa también los permisos del token: debe permitir leer ese modelo gated. '
                'Un token válido y el acceso al modelo son comprobaciones distintas.'
            ) from None
        raise RuntimeError(f'No se pudo comprobar el acceso al modelo (HTTP {status}); reintenta más tarde.') from None
    except RequestException:
        raise RuntimeError('Error de conexión al comprobar los permisos del modelo. Reintenta.') from None
    print('Acceso de lectura a DINOv3: OK')
    return token

HF_TOKEN = verify_hf_access(globals().get('HF_TOKEN'), MODEL_ID)

manifest_path = RUN_ROOT / 'config.json'
old_manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else None
try:
    # Reuse the same model revision on every resume, even if upstream main changes.
    revision = (old_manifest['model_revision'] if old_manifest else HfApi(token=HF_TOKEN).model_info(MODEL_ID).sha)
    CFG['model_revision'] = revision
    reference_processor = AutoImageProcessor.from_pretrained(
        MODEL_ID, revision=revision, token=HF_TOKEN, trust_remote_code=False)
except (OSError, ValueError) as exc:
    raise RuntimeError('No se pudo acceder al checkpoint. Abre su página de Hugging Face, solicita/acepta el acceso '
                       'y usa HF_TOKEN de esa misma cuenta. No se cargan pesos aleatorios.') from exc

reference = dict(size=reference_processor.size, mean=list(reference_processor.image_mean),
                 std=list(reference_processor.image_std), resample=int(reference_processor.resample))
np.testing.assert_allclose(reference['mean'], CFG['preprocessing']['mean'])
np.testing.assert_allclose(reference['std'], CFG['preprocessing']['std'])
CFG['hf_processor_reference'] = reference
print('Checkpoint:', MODEL_ID)
print('Revisión fijada:', revision)
print('Procesador HF declara:', reference)
print('Entrada elegida explícitamente: 256×256, bilinear, según README oficial; no se ejecuta un segundo resize.')

def load_backbone():
    loaded, info = AutoModel.from_pretrained(
        MODEL_ID, revision=CFG['model_revision'], token=HF_TOKEN,
        use_safetensors=True, trust_remote_code=False, torch_dtype=torch.float32,
        output_loading_info=True,
    )
    errors = {key: info.get(key) for key in ('missing_keys', 'unexpected_keys', 'mismatched_keys', 'error_msgs') if info.get(key)}
    if errors:
        raise RuntimeError(f'Carga de pesos incompleta: {errors}')
    if loaded.config.model_type != 'dinov3_convnext' or loaded.config.hidden_sizes[-1] != CFG['feature_dim']:
        raise ValueError('El checkpoint no es el DINOv3 ConvNeXt Base esperado.')
    loaded.requires_grad_(False)
    return loaded.to(DEVICE, memory_format=torch.channels_last).eval()

backbone = load_backbone()
print(f'CNN cargada: {sum(p.numel() for p in backbone.parameters()):,} parámetros. Todos congelados para la primera fase.')

In [ ]:
import kagglehub
if not os.environ.get('KAGGLE_API_TOKEN'):
    kaggle_token = colab_secret('KAGGLE_API_TOKEN')
    if kaggle_token:
        os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    del kaggle_token
kagglehub.login()
download_root = Path(kagglehub.competition_download('mva-recvis-2026'))
data_files = {}
for filename in ('train.npz', 'validation.npz', 'test.npz', 'sample_submission.csv'):
    matches = list(download_root.rglob(filename))
    if len(matches) != 1:
        raise RuntimeError(f'Esperaba un {filename}; encontrados: {matches}')
    data_files[filename] = matches[0]

def read_labeled(path):
    with np.load(path, allow_pickle=False) as data:
        images, labels = data['images'], data['labels']
    assert images.dtype == np.uint8 and images.shape[1:] == (32, 32, 3)
    assert np.issubdtype(labels.dtype, np.integer)
    labels = labels.reshape(-1).astype(np.int64)
    assert len(images) == len(labels) > 0 and ((0 <= labels) & (labels < 10)).all()
    return images, labels

train_images, train_labels = read_labeled(data_files['train.npz'])
val_images, val_labels = read_labeled(data_files['validation.npz'])
with np.load(data_files['test.npz'], allow_pickle=False) as data:
    test_images, test_ids = data['images'], data['ids']
assert test_images.dtype == np.uint8 and test_images.shape[1:] == (32, 32, 3)
assert test_ids.ndim == 1 and len(test_ids) == len(test_images)
assert np.issubdtype(test_ids.dtype, np.integer) and len(np.unique(test_ids)) == len(test_ids)
# Only public test pixels and IDs are read; never labels.
CFG['data_sha256'] = {name: sha256_file(path) for name, path in data_files.items()}
if old_manifest is not None and old_manifest != CFG:
    raise ValueError('Configuración/datos distintos: cambia RUN_NAME para iniciar otro experimento.')
if not manifest_path.exists():
    manifest_path.write_text(json.dumps(CFG, indent=2))
print('Train:', train_images.shape, '| validación:', val_images.shape, '| test:', test_images.shape)

In [ ]:
# Official README order: uint8 tensor -> bilinear resize -> float [0,1] -> normalize.
preprocess = v2.Compose([
    v2.ToImage(),
    v2.Resize((256, 256), interpolation=InterpolationMode.BILINEAR, antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(CFG['preprocessing']['mean'], CFG['preprocessing']['std']),
])
basic_transform = T.Compose([
    T.RandomCrop(32, padding=CFG['crop_padding'], padding_mode='reflect'),
    T.RandomHorizontalFlip(),
])
augmix_transform = T.AugMix(severity=CFG['augmix_severity'], mixture_width=CFG['augmix_width'],
                           chain_depth=-1, alpha=1.0, all_ops=True,
                           interpolation=InterpolationMode.BILINEAR)
sample = preprocess(Image.fromarray(train_images[0]))
assert sample.shape == (3, 256, 256) and sample.dtype == torch.float32 and torch.isfinite(sample).all()
val_loader = make_loader(ImagesDataset(val_images, val_labels, preprocess))
test_loader = make_loader(ImagesDataset(test_images, test_ids, preprocess))

with torch.inference_mode(), torch.autocast('cuda', dtype=AMP_DTYPE):
    sample_embedding = embedding(backbone, sample.unsqueeze(0).to(DEVICE))
assert sample_embedding.shape == (1, 1024) and torch.isfinite(sample_embedding).all()
print('Entrada comprobada:', tuple(sample.shape), '| descriptor global:', tuple(sample_embedding.shape))
del sample, sample_embedding

## 3. CNN congelada y clasificador lineal

Extraemos una vez las características de train original, train volteado y validación. Solo la cabeza ve las etiquetas y recibe gradientes. La caché se conserva en Drive. Las 100 épocas máximas son de la cabeza, **no 100 pasadas de entrenamiento por ConvNeXt**.

In [ ]:
train_original = extract_features(backbone, train_images, 'train_original', preprocess)
train_flipped = extract_features(backbone, train_images, 'train_flip', preprocess, flip=True)
validation_features = extract_features(backbone, val_images, 'validation', preprocess)
linear_train_features = torch.cat([train_original, train_flipped], dim=0)
linear_train_labels = torch.from_numpy(np.tile(train_labels, 2))
linear_head, linear_summary = train_linear(linear_train_features, linear_train_labels,
                                           validation_features, torch.from_numpy(val_labels))
display(pd.DataFrame([linear_summary]))
del linear_train_features, linear_train_labels, train_original, train_flipped, validation_features
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Produce this CSV before fine-tuning, preserving a result if the runtime disconnects later.
test_features = extract_features(backbone, test_images, 'test', preprocess)
linear_predictions = predict_features(linear_head, test_features)
write_submission(test_ids, linear_predictions, data_files['sample_submission.csv'],
                 RUN_ROOT / 'submission_dinov3_linear_256.csv')
del test_features
gc.collect()

## 4. Ajuste parcial desde el clasificador anterior

Se vuelve a cargar la CNN original desde la revisión fijada y la mejor cabeza de la fase 3. Se desbloquea únicamente **la cuarta etapa y la LayerNorm final**. Las tres primeras etapas quedan congeladas. No hay BatchNorm ni recalibración con el test.

La pérdida es CE de la vista básica + 6 × JSD entre esa vista y dos AugMix suaves. Los aumentos se producen a 32×32, y cada vista se amplía una sola vez a 256. Esto es adaptación supervisada de nuestro clasificador, **no repetir DINO ni destilar nuevas etiquetas**.

También se evalúa el modelo antes del primer paso y se guarda como **época 0**. Si ninguna época mejora ese punto según validación, el resultado seleccionado seguirá siendo la época 0; no significa que el ajuste haya mejorado. El primer CSV siempre se conserva para comparar en Kaggle.

In [ ]:
# Each call reconstructs frozen layers from the original checkpoint before loading adaptation state.
if 'ft_model' in globals():
    del ft_model
    gc.collect()
    torch.cuda.empty_cache()
ft_backbone = load_backbone()
saved_head = torch.load(RUN_ROOT / 'linear' / 'best.pt', map_location='cpu', weights_only=True)
ft_head = nn.Linear(CFG['feature_dim'], 10).to(DEVICE)
ft_head.load_state_dict(saved_head['head'])
del saved_head
ft_model = DinoClassifier(ft_backbone, ft_head)
del ft_backbone, ft_head
finetuning_summary = finetune(ft_model, train_images, train_labels, val_loader,
                             preprocess, basic_transform, augmix_transform)
display(pd.DataFrame([finetuning_summary]))

In [ ]:
prediction_ids, finetuned_predictions = predict_images(ft_model, test_loader)
write_submission(prediction_ids, finetuned_predictions, data_files['sample_submission.csv'],
                 RUN_ROOT / 'submission_dinov3_partial_ft_256.csv')

## 5. Comparación y descarga

Los resultados DINOv3 se rellenan al ejecutar. Sube los **dos CSV individualmente** a Kaggle y copia sus scores en la celda siguiente. Las filas ResNet son las referencias comunicadas previamente, no nuevos entrenamientos. `training_hours` mide las épocas: excluye descarga, extracción de características e inferencia; el coste de extraer características se muestra durante la ejecución.

El test sintético y la validación CIFAR-10 difieren. No inferimos la accuracy del test a partir de confianza, pérdida o desacuerdo. La comparación entre ResNet y DINO cambia arquitectura, preentrenamiento, resolución y adaptación; no es una ablación aislada de arquitectura.

In [ ]:
KAGGLE_SCORES = {'dinov3_linear': None, 'dinov3_partial_ft': None}
rows = [
    dict(method='resnet18_baseline_128', val_accuracy=0.9604, kaggle_accuracy=0.61093,
         best_epoch=29, provenance='reported_by_user'),
    dict(method='resnet18_augmix_128', val_accuracy=0.9656, kaggle_accuracy=0.75796,
         best_epoch=29, provenance='reported_by_user'),
]
for folder, method in (('linear', 'dinov3_linear'), ('partial_ft', 'dinov3_partial_ft')):
    row = json.loads((RUN_ROOT / folder / 'summary.json').read_text())
    row['kaggle_accuracy'] = KAGGLE_SCORES[method]
    row['provenance'] = 'this_run'
    rows.append(row)
comparison = pd.DataFrame(rows)
comparison.to_csv(RUN_ROOT / 'comparison.csv', index=False)
display(comparison.style.format({'val_accuracy': '{:.2%}', 'kaggle_accuracy': '{:.2%}',
                                'training_hours': '{:.2f}'}, na_rep='pendiente'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, folder, title in zip(axes, ('linear', 'partial_ft'), ('Cabeza lineal', 'Ajuste parcial')):
    history = pd.read_csv(RUN_ROOT / folder / 'history.csv')
    ax.plot(history.epoch, 100 * history.val_accuracy, marker='.')
    summary = json.loads((RUN_ROOT / folder / 'summary.json').read_text())
    ax.axvline(summary['best_epoch'], color='gray', linestyle='--', label='checkpoint elegido')
    if folder == 'partial_ft':
        ax.axhline(100 * summary['initial_val_accuracy'], color='orange', linestyle=':', label='antes de ajustar')
    ax.set(title=title, xlabel='Época', ylabel='Validación limpia (%)')
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(RUN_ROOT / 'validation_curves.png', dpi=150)
plt.show()
lp = pd.read_csv(RUN_ROOT / 'submission_dinov3_linear_256.csv').set_index('id')
ft = pd.read_csv(RUN_ROOT / 'submission_dinov3_partial_ft_256.csv').set_index('id').reindex(lp.index)
print(f'Desacuerdo de predicciones: {(lp.label != ft.label).mean():.2%} (no es accuracy).')

In [ ]:
bundle = RUN_ROOT / 'dinov3_two_submissions.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for filename in ('submission_dinov3_linear_256.csv', 'submission_dinov3_partial_ft_256.csv',
                     'comparison.csv', 'validation_curves.png', 'config.json'):
        archive.write(RUN_ROOT / filename, arcname=filename)
    for folder in ('linear', 'partial_ft'):
        archive.write(RUN_ROOT / folder / 'history.csv', arcname=f'{folder}_history.csv')
from google.colab import files
files.download(str(bundle))
print('Descomprime y sube cada submission_*.csv por separado a Kaggle. Los pesos permanecen en Drive.')

## Actualización desde GitHub (opcional)
Esta celda muestra un enlace a la última revisión. Las copias anteriores de Drive no incorporan cambios automáticamente. Abre el enlace para usar el código nuevo; el estado en memoria de esta pestaña no se traslada. Los checkpoints compatibles se recuperan de Drive.

In [ ]:
import re
import time
import urllib.request
from IPython.display import HTML
try:
    request = urllib.request.Request(
        'https://api.github.com/repos/rivas-vega/A3_Recv/commits/main?t=' + str(time.time_ns()),
        headers={'Accept': 'application/vnd.github+json', 'User-Agent': 'A3-Recv-Colab'})
    with urllib.request.urlopen(request, timeout=20) as response:
        latest_sha = json.load(response)['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', latest_sha)
    url = f'https://colab.research.google.com/github/rivas-vega/A3_Recv/blob/{latest_sha}/main_notebook.ipynb'
    display(HTML(f'<a href="{url}" target="_blank" rel="noopener noreferrer">Abrir última versión ({latest_sha[:7]})</a>'))
except Exception as exc:
    print('No se pudo consultar GitHub:', exc)

Para el informe declara el checkpoint, revisión, preentrenamiento web y destilación previa desde ViT-7B; entrada bilineal 256, cabeza lineal, capas desbloqueadas, LR, aumentos, JSD y criterio de selección. No atribuyas una eventual mejora solo a la arquitectura CNN. Los resultados dependen también del preentrenamiento y adaptación.

El repositorio es público: no guardes tokens, datos, cachés, pesos ni salidas con imágenes. El ZIP de resultados es distinto del ZIP de código de la entrega (máximo 8 MB, sin pesos/datos). Este notebook se publica sin salidas y sin resultados DINOv3 inventados.